Problems: need to strip the start of name (has professor and stuff)
some profs dont exist? but ill still include (its Dr Chris Bell)
Some people have the title to not actually be their prof level, instead its in their name



In [13]:
#pip install requests beautifulsoup4

In [14]:
import requests

url = "https://business.uq.edu.au/team/finance-discipline"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=10)

In [15]:
print(response.status_code)   # want 200
print(len(response.text))     # want something large, ~50k+ for these pages

200
94917


In [16]:
with open("page.html", "w", encoding="utf-8") as f:
    f.write(response.text)

In [17]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")
cards = soup.select(".person--teaser")
print(len(cards))

24


In [18]:
print(cards[0].prettify())

<div class="person--teaser">
 <div class="row profile__wrapper">
  <div class="person">
   <div class="column small-5 medium-3 profile__media">
    <div class="person__photo">
     <img src="/sites/all/themes/custom/uq_standard/images/profile-placeholder.png">
     </img>
    </div>
   </div>
   <div class="column small-7 medium-9 profile__content">
    <h3 class="person__display-name">
     <a href="/profile/9542/jon-aster">
      Mr Jon Aster
     </a>
    </h3>
    <div class="person__position">
     <div class="position__title">
      Associate Lecturer
     </div>
     <div class="position__organisation">
      School of Business
     </div>
    </div>
   </div>
  </div>
 </div>
</div>



In [19]:
print(cards[0].select_one(".person__display-name a").get_text(strip=True))

Mr Jon Aster


In [20]:
print(cards[0].select_one(".position__title").get_text(strip=True))

Associate Lecturer


In [21]:
import re

PREFIX = re.compile(
    r"^(Associate Professor|Emeritus Professor|Professor|Dr|Mr|Mrs|Ms|Miss|A/Prof|Prof|Assoc\.? Prof\.?)\.?\s+",
    re.IGNORECASE
)



In [22]:
import requests, time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

import re

PREFIX = re.compile(
    r"^(Associate Professor|Emeritus Professor|Professor|Dr|Mr|Mrs|Ms|Miss|A/Prof|Prof|Assoc\.? Prof\.?)\.?\s+",
    re.IGNORECASE
)

SUFFIX = re.compile(r"\s*\([^)]*\)\s*$")



def alive(url):
    try:
        return requests.head(url, allow_redirects=True, timeout=10).status_code == 200
    except requests.RequestException:
        return False

# ... fetch and parse ...

# loop 1 — extraction only, no network
records = []
for card in cards:
    link = card.select_one(".person__display-name a")
    if not link:
        continue
    name = link.get_text(strip=True)
    m = PREFIX.match(name)
    href = link["href"]

    titles = [t.get_text(strip=True) for t in card.select(".position__title")]
    titles = [t for t in titles if t]
    substantive = [t for t in titles if not t.startswith("Affiliate")]
    title = (substantive or titles or [None])[0]

    records.append({
    "name": name,
    "name_clean": PREFIX.sub("", name).strip(),
    "prefix": m.group(1) if m else None,
    "title": title,
    "title_clean": SUFFIX.sub("", title).strip() if title else None,
    "profile_url": urljoin("https://business.uq.edu.au", href),
})

print(len(records))  # expect 24

# loop 2 — network checks
for r in records:
    r["alive"] = alive(r["profile_url"])
    time.sleep(1)

24


In [ ]:
KEEP = ["name_clean", "title_clean", "profile_url"]
records = [{k: r[k] for k in KEEP} for r in records]
records

[{'name_clean': 'Jon Aster',
  'title_clean': 'Associate Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/9542/jon-aster'},
 {'name_clean': 'Chris Bell',
  'title_clean': None,
  'profile_url': 'https://business.uq.edu.au/profile/17730/chris-bell'},
 {'name_clean': 'Shaun Bond',
  'title_clean': 'Professor',
  'profile_url': 'https://business.uq.edu.au/profile/6239/shaun-bond'},
 {'name_clean': 'Alexander Cameron',
  'title_clean': 'Associate Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/18606/alexander-cameron'},
 {'name_clean': 'Yong Ming Chen',
  'title_clean': 'Teaching Associate',
  'profile_url': 'https://business.uq.edu.au/profile/12939/yong-ming-chen'},
 {'name_clean': 'Hasibul Chowdhury',
  'title_clean': 'Senior Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/1865/hasibul-chowdhury'},
 {'name_clean': 'Nicolas Eugster',
  'title_clean': 'Senior Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/8977/nicolas-eugster'

In [ ]:
#I dont know what Stephen Gray